In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import time
import datetime

spark = SparkSession.builder \
    .appName("ContentBased_Filtering_Research") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()


In [2]:
# Tải dữ liệu từ tầng Gold
events_df = spark.table("gold_db.fact_events")
products_df = spark.table("gold_db.dim_products")

# Chia tập dữ liệu 
max_date = events_df.select(F.max("event_time")).first()[0]
cutoff = max_date - datetime.timedelta(days=30)

train_events = events_df.filter(F.col("event_time") < cutoff)
test_events = events_df.filter(F.col("event_time") >= cutoff)

# Lấy lịch sử tương tác của User trong tập Train để biết user thích cái gì
user_history = train_events.groupBy("user_id", "product_id").agg(F.count("event_type").alias("interact_count"))

print(f"Dữ liệu sẵn sàng. Số lượng sản phẩm trong kho: {products_df.count()}")

Dữ liệu sẵn sàng. Số lượng sản phẩm trong kho: 53453


In [3]:
# Logic: Với mỗi sản phẩm, tìm các sản phẩm khác "giống" nó nhất
# Tiêu chí: Cùng category_detail (+3 điểm), Cùng category_sub (+2 điểm), Cùng brand (+1 điểm)
start_time = time.time()

# Join bảng sản phẩm với chính nó (Self-join) để tìm cặp tương đồng
# giới hạn join trong cùng Category_Main để tránh nổ bộ nhớ
item_similarities = products_df.alias("p1").join(
    products_df.alias("p2"),
    (F.col("p1.category_main") == F.col("p2.category_main")) & (F.col("p1.product_id") != F.col("p2.product_id"))
).select(
    F.col("p1.product_id").alias("product_id_1"),
    F.col("p2.product_id").alias("product_id_2"),
    (
        F.when(F.col("p1.category_detail") == F.col("p2.category_detail"), 3).otherwise(0) +
        F.when(F.col("p1.category_sub") == F.col("p2.category_sub"), 2).otherwise(0) +
        F.when(F.col("p1.brand") == F.col("p2.brand"), 1).otherwise(0)
    ).alias("content_score")
).filter("content_score > 0")

# Lấy Top 20 sản phẩm tương đồng nhất cho mỗi sản phẩm
window_spec = Window.partitionBy("product_id_1").orderBy(F.desc("content_score"))
top_sim_items = item_similarities.withColumn("rank", F.row_number().over(window_spec)).filter("rank <= 20")

training_time = time.time() - start_time 

# Lưu vào Gold để sau này Web gọi "Sản phẩm tương tự"
top_sim_items.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("path", "s3a://gold/recommendations_content_based") \
    .saveAsTable("gold_db.recommendations_content_based")

In [4]:
# Tạo gợi ý cho User
# Join lịch sử User với bảng tương đồng nội dung vừa tính
recommendations = user_history.join(
    top_sim_items, user_history.product_id == top_sim_items.product_id_1
).select(
    "user_id", 
    F.col("product_id_2").alias("product_id"), 
    F.col("content_score").alias("prediction")
).groupBy("user_id", "product_id").agg(F.sum("prediction").alias("prediction"))

# ĐÁNH GIÁ (So sánh với tập Test)
actual_items = test_events.groupBy("user_id").agg(F.collect_set("product_id").alias("actual_list"))

# Lấy Top 10 cho mỗi user
window_rank = Window.partitionBy("user_id").orderBy(F.desc("prediction"))
top_10_recs = recommendations.withColumn("rank", F.row_number().over(window_rank)).filter("rank <= 10")
predicted_items = top_10_recs.groupBy("user_id").agg(F.collect_set("product_id").alias("pred_list"))

# Tính Precision, Recall, F1
comparison = actual_items.join(predicted_items, "user_id")
match_df = comparison.withColumn("hits", F.size(F.array_intersect("actual_list", "pred_list")))

precision_at_10 = match_df.select(F.avg(F.col("hits") / 10)).first()[0]
recall_at_10 = match_df.select(F.avg(F.col("hits") / F.size("actual_list"))).first()[0]
f1_val = 2 * (precision_at_10 * recall_at_10) / (precision_at_10 + recall_at_10) if (precision_at_10 + recall_at_10) > 0 else 0

print("\n" + "="*85)
print(f"{'Mô hình':<15} | {'MAPE':<10} | {'RMSE':<10} | {'Prec@10':<10} | {'Rec@10':<10} | {'F1':<10} | {'Time':<10}")
print("-" * 85)
print(f"{'Content-Based':<15} | {'N/A':>10} | {'N/A':>10} | {precision_at_10:>10.4f} | {recall_at_10:>10.4f} | {f1_val:>10.4f} | {training_time:>8.2f}s")
print("="*85 + "\n")


Mô hình         | MAPE       | RMSE       | Prec@10    | Rec@10     | F1         | Time      
-------------------------------------------------------------------------------------
Content-Based   |        N/A |        N/A |     0.0032 |     0.0166 |     0.0054 |     0.11s

